# ETL Orchestration — run the whole chain

Runs all six stages in order via `pipelines.run_all`. Each stage is still independent (they communicate only through the shared artifact store); this notebook is just the convenience path. For working on a single stage, open its own notebook (`00_ingest.ipynb` … `50_metrics.ipynb`).

**Stage map**

| Stage | Owner | Concept added |
|---|---|---|
| 00 ingest | Data Engineer | — |
| 10 pd | ML Engineer | — |
| 20 graph | Data Scientist · graph | **arpym #2: MP spectrum denoising** |
| 30 copula | Data Scientist · copula | — |
| 40 transitions | Risk Analyst / Credit Risk | **arpym #1: generator-based estimator** |
| 50 metrics | Risk Analyst | — |


## 1. Setup

In [ ]:
# --- setup: make the project root importable + open the shared artifact store ---
import sys, os
ROOT = os.path.abspath("..")          # notebooks/ lives one level below the project root
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from pipelines import ArtifactStore
store = ArtifactStore("output/etl")   # the SAME store every stage reads/writes
print("artifact store:", store.root)
print("existing artifacts:", store.list())


## 2. Run the full pipeline

`denoise=True` turns on the Marčenko-Pastur denoising in stage 20. Use `stage_opts` to pass per-stage overrides.

In [ ]:
from pipelines import run_all

store = run_all(
    root="output/etl",
    seed=42,
    denoise=True,
    stage_opts={
        "30_copula": {"copula_type": "clayton"},
        "40_transitions": {"tau_hl_years": 2.0},
        "50_metrics": {"segment_cols": ["city_name", "risk_archetype"]},
    },
    verbose=True,
)


## 3. All artifacts produced

In [ ]:
for a in store.list():
    print(a)
